# 🔍 Zane Holloway — The Street Detective | Voice Aging MVP

**Character**: Zane Holloway — brilliant, cynical investigator. Starts eager, ends burned out but still going.

**Core Personality**: `a sharp, sardonic male detective — fast-thinking, street-smart, voice like gravel and sarcasm, hides genuine moral conviction behind a wall of cynicism`

## 📈 Age Stages
| Stage | Age | Title | Voice Modifier |
|---|---|---|---|
| Youth | 23 | The Rookie | eager and fast-talking, genuine enthusiasm before cynicism sets in, nervous energy, still idealistic |
| Prime | 37 | The Ace | perfectly controlled sardonic wit, gravel settling into voice, sharp and unhurried, dangerously perceptive |
| Middle | 51 | The Worn | heavier and slower, the sardonic wit still present but quieter, exhaustion visible, something like pain underneath the control |
| Elder | 68 | The Legend | gravelly and slow, wry humor intact but softer, exhausted authority, knows things no one else knows |

In [ ]:
# 🛠️ 1. Setup & Imports
!pip install -q git+https://github.com/huggingface/transformers accelerate soundfile librosa
!pip install -q qwen-tts

import os
import gc
import torch
import numpy as np
import soundfile as sf
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel

OUTPUT_DIR = "/content/zane_holloway_aging"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Created output directory at: {OUTPUT_DIR}")

In [ ]:
# 🎭 2. Character Data
CHARACTER = {
    "name": "Zane Holloway",
    "personality_core": "a sharp, sardonic male detective — fast-thinking, street-smart, voice like gravel and sarcasm, hides genuine moral conviction behind a wall of cynicism",
    "stages": [
        {
            "stage_name": "Youth (23) — The Rookie",
            "voice_modifier": "eager and fast-talking, genuine enthusiasm before cynicism sets in, nervous energy, still idealistic",
            "prefix": "zane_holloway_youth",
            "lines": [
                "I joined the force to solve things. Puzzles. Cases. People. Turns out they're all the same problem.",
                "Give me forty-eight hours and any crime scene and I'll tell you exactly what happened. That's not arrogance. That's just been my experience so far.",
                "Everyone has a tell. Everyone. You just have to know what you're looking for."
            ]
        },
        {
            "stage_name": "Prime (37) — The Ace",
            "voice_modifier": "perfectly controlled sardonic wit, gravel settling into voice, sharp and unhurried, dangerously perceptive",
            "prefix": "zane_holloway_prime",
            "lines": [
                "I've been lied to by professionals. Politicians. Killers. CEOs. You're going to have to do considerably better than this.",
                "The trick isn't finding the guilty party. Every case has one of those. The trick is building something a jury can't ignore.",
                "I don't trust my instincts. I test them. Then I test them again. Then I trust them."
            ]
        },
        {
            "stage_name": "Middle (51) — The Worn",
            "voice_modifier": "heavier and slower, the sardonic wit still present but quieter, exhaustion visible, something like pain underneath the control",
            "prefix": "zane_holloway_middle",
            "lines": [
                "I've solved four hundred and twelve cases. I remember every single one. Especially the twelve I didn't solve.",
                "At some point you stop being surprised by what people do to each other. That's not peace. That's just damage.",
                "I keep doing this because stopping would mean admitting the twelve cases won. I'm not ready for that."
            ]
        },
        {
            "stage_name": "Elder (68) — The Legend",
            "voice_modifier": "gravelly and slow, wry humor intact but softer, exhausted authority, knows things no one else knows",
            "prefix": "zane_holloway_elder",
            "lines": [
                "Everyone who tried to warn me this job would break me was right. They just underestimated how long it would take.",
                "The best investigators aren't the smartest ones. They're the ones who care enough to keep going when the smart ones quit.",
                "Forty-five years. One rule: find the truth. Everything else — the politics, the paperwork, the politics again — is noise."
            ]
        }
    ]
}

In [ ]:
# 🧠 3. Load Qwen3-TTS VoiceDesign Model
gc.collect()
torch.cuda.empty_cache()

model_id = "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign"
print(f"Loading {model_id}...")

model = Qwen3TTSModel.from_pretrained(
    model_id,
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

print("Model loaded successfully!")

In [ ]:
# 🎙️ 4. Generate Aging Stages
print(f"Generating all age stages for {CHARACTER['name']}...\n")

for stage in CHARACTER["stages"]:
    print(f"=== {stage['stage_name']} ===")
    instruct = f"{CHARACTER['personality_core']}, {stage['voice_modifier']}"
    print(f"Voice Design Prompt:\n{instruct}\n")
    
    for i, line in enumerate(stage["lines"], 1):
        filename = f"{stage['prefix']}_{i:02d}.wav"
        filepath = os.path.join(OUTPUT_DIR, filename)
        
        print(f"Line {i}: {line}")
        
        # Generate voice
        wavs, sr = model.generate_voice_design(line, "English", instruct)
        
        # Save and display
        sf.write(filepath, wavs[0], sr)
        display(Audio(filepath))
        
        gc.collect()
        torch.cuda.empty_cache()
    print("-" * 40)

In [ ]:
# 🎬 5. Life Story Montage
print("=== Life Story Montage ===")
montage_audio = []
script = []
current_sr = 24000

for stage in CHARACTER["stages"]:
    line = stage["lines"][0] # Pick Line 1 from each stage
    instruct = f"{CHARACTER['personality_core']}, {stage['voice_modifier']}"
    
    wavs, sr = model.generate_voice_design(line, "English", instruct)
    current_sr = sr
    
    # 2.5 seconds of silence gap
    silence = np.zeros(int(2.5 * sr), dtype=np.float32)
    
    montage_audio.extend([wavs[0], silence])
    script.append(f"[{stage['stage_name']}] {line}")

final_audio = np.concatenate(montage_audio)
montage_path = os.path.join(OUTPUT_DIR, "zane_holloway_life_story.wav")
sf.write(montage_path, final_audio, current_sr)

print("\n📜 Reading Script:")
for s in script:
    print(s)

print("\n▶️ Play Montage:")
display(Audio(montage_path))

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 🎭 6. Emotional Range
target_line = "I've been lied to by professionals. Politicians. Killers. CEOs. You're going to have to do considerably better than this."
prime_modifier = CHARACTER["stages"][1]["voice_modifier"]
base_instruct = f"{CHARACTER['personality_core']}, {prime_modifier}"

emotions = [
    {
        "name": "Amused Contempt",
        "mod": "[amused contempt]"
    },
    {
        "name": "Quiet Anger",
        "mod": "[quiet anger]"
    },
    {
        "name": "Tired Disappointment",
        "mod": "[tired disappointment]"
    }
]

print("=== Emotional Range ===")
print(f"Line: \"{target_line}\"\n")

for emo in emotions:
    print(f"▶️ Register: {emo['name']}")
    instruct = f"{base_instruct}, {emo['mod']}"
    
    wavs, sr = model.generate_voice_design(target_line, "English", instruct)
    
    filename = f"zane_holloway_prime_{emo['name'].replace(' ', '_').lower()}.wav"
    filepath = os.path.join(OUTPUT_DIR, filename)
    sf.write(filepath, wavs[0], sr)
    display(Audio(filepath))
    
    gc.collect()
    torch.cuda.empty_cache()
    print()

In [ ]:
# 📦 7. Download Outputs
import shutil
from google.colab import files

print("Zipping outputs...")
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print("Downloading...")
files.download(f"{OUTPUT_DIR}.zip")